# CatBoost Optimization: Feature Engineering + Tuning

In [1]:
# Setup & load data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 250)

RANDOM_STATE = 42
TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")
train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="catboost")

TARGET = cfg.target_col
ID = cfg.id_col

y = train[TARGET]
X = train.drop(columns=[TARGET])

In [2]:
# Make CatBoost-safe copies (categorical NaNs → "missing")

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in X.columns if not is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if is_numeric_dtype(X[c])]

X_cb = X.copy()
test_cb = test.copy()

for c in cat_cols:
    X_cb[c] = X_cb[c].astype("string").fillna("missing")
    test_cb[c] = test_cb[c].astype("string").fillna("missing")

# IMPORTANT: categorical indices for Pool
cat_feature_indices = [X_cb.columns.get_loc(c) for c in cat_cols]

print("cat:", len(cat_cols), "num:", len(num_cols))

cat: 32 num: 14


In [12]:
# Feature engineering function (train + test consistent)

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ---------
    # 1) Finance Access Score
    # ---------
    access_cols = [c for c in [
        "has_credit_card",
        "has_debit_card",
        "has_loan_account",
        "has_internet_banking",
        "has_mobile_money",
    ] if c in df.columns]

    # Convert yes/no-ish strings into 1/0 safely
    def yn_to01(s):
        s = s.astype("string").str.lower().fillna("missing")
        return s.isin(["yes", "y", "1", "true"]).astype(int)

    for c in access_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if access_cols:
        df["access_score"] = df[[f"{c}__01" for c in access_cols]].sum(axis=1)

    # ---------
    # 2) Insurance Coverage Score
    # ---------
    insurance_cols = [c for c in [
        "funeral_insurance",
        "medical_insurance",
        "motor_vehicle_insurance",
        "has_insurance",
    ] if c in df.columns]

    for c in insurance_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if insurance_cols:
        df["insurance_score"] = df[[f"{c}__01" for c in insurance_cols]].sum(axis=1)

    # ---------
    # 3) Resilience / Stress Score
    # ---------
    stress_cols = [c for c in [
        "current_problem_cash_flow",
        "attitude_worried_shutdown",
        "problem_sourcing_money",
    ] if c in df.columns]

    for c in stress_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if stress_cols:
        df["stress_score"] = df[[f"{c}__01" for c in stress_cols]].sum(axis=1)

    # ---------
    # 4) Financial Records / Compliance Score
    # ---------
    records_cols = [c for c in [
        "keeps_financial_records",
        "compliance_income_tax",
    ] if c in df.columns]

    for c in records_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if records_cols:
        df["formalization_score"] = df[[f"{c}__01" for c in records_cols]].sum(axis=1)

    # ---------
    # 5) Business scale ratios (only if numeric columns exist)
    # ---------
    if "business_turnover" in df.columns and "business_expenses" in df.columns:
        turn = pd.to_numeric(df["business_turnover"], errors="coerce")
        exp  = pd.to_numeric(df["business_expenses"], errors="coerce")
        df["turnover_minus_expenses"] = turn - exp
        df["turnover_to_expenses"] = turn / (exp.replace(0, np.nan))
        df["turnover_to_expenses"] = df["turnover_to_expenses"].replace([np.inf, -np.inf], np.nan)

    # ---------
    # 6) Age bucket (helps country/market nonlinearity)
    # ---------
    if "business_age_total_months" in df.columns:
        age = pd.to_numeric(df["business_age_total_months"], errors="coerce")
        df["business_age_years"] = (age / 12.0)
        age_bucket = pd.cut(
        df["business_age_years"],
        bins=[-np.inf, 0.5, 2, 5, 10, np.inf],
        labels=["<6m", "0.5-2y", "2-5y", "5-10y", "10y+"]
        )
        df["age_bucket"] = age_bucket.astype("string").fillna("missing")

    # ---------
    # Cleanup engineered ratio NaNs (CatBoost handles numeric NaNs OK)
    # ---------
    return df

In [13]:

# Apply it
X_fe = add_features(X_cb)
test_fe = add_features(test_cb)

# Convert pandas NA to numpy nan everywhere (CatBoost-friendly)
X_fe = X_fe.replace({pd.NA: np.nan})
test_fe = test_fe.replace({pd.NA: np.nan})

# Recompute cat cols/indices after new features (important!)
from pandas.api.types import is_numeric_dtype
cat_cols_fe = [c for c in X_fe.columns if not is_numeric_dtype(X_fe[c])]
cat_feature_indices_fe = [X_fe.columns.get_loc(c) for c in cat_cols_fe]

print("Before:", X_cb.shape, "After FE:", X_fe.shape)



Before: (9618, 46) After FE: (9618, 67)


In [14]:
# CatBoost tuning strategy
# CV runner (macro F1 + OOF)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def catboost_cv_score(X, y, cat_idx, params, n_splits=5, seed=42, verbose_eval=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.empty(len(y), dtype=object)
    scores = []

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        tr_pool = Pool(X.iloc[tr], y.iloc[tr], cat_features=cat_idx)
        va_pool = Pool(X.iloc[va], y.iloc[va], cat_features=cat_idx)

        model = CatBoostClassifier(**params)
        model.fit(tr_pool, eval_set=va_pool, use_best_model=True, verbose=verbose_eval)

        pred = model.predict(va_pool)
        pred = np.asarray(pred).reshape(-1).astype(str)

        oof[va] = pred
        f1 = macro_f1(y.iloc[va], pred)
        scores.append(f1)

        print(f"Fold {fold}: {f1:.4f}")

    mean = float(np.mean(scores))
    std = float(np.std(scores))
    oof_f1 = macro_f1(y, oof)

    print(f"Mean: {mean:.4f} ± {std:.4f} | OOF: {oof_f1:.4f}")
    return mean, std, oof_f1, scores

In [15]:
# Define a better baseline param set (faster than 5000 iters)
# From earlier training: best iterations were often < 500.
# So for tuning:
#   set iterations=2000
#   early stop 100
#   use thread_count=-1 for speed

base_params = {
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "random_seed": RANDOM_STATE,
    "iterations": 2000,
    "early_stopping_rounds": 100,
    "learning_rate": 0.06,
    "depth": 6,
    "l2_leaf_reg": 6.0,
    "subsample": 0.9,
    "bootstrap_type": "Bernoulli",
    "min_data_in_leaf": 20,
    "thread_count": -1,
    "task_type": "CPU",
    "verbose": False
}


In [16]:
# Hyperparameter candidates (small, high-impact search)

search_space = [
    {"depth": 6, "learning_rate": 0.06, "l2_leaf_reg": 6.0, "min_data_in_leaf": 20, "subsample": 0.9},
    {"depth": 7, "learning_rate": 0.05, "l2_leaf_reg": 8.0, "min_data_in_leaf": 25, "subsample": 0.9},
    {"depth": 5, "learning_rate": 0.08, "l2_leaf_reg": 5.0, "min_data_in_leaf": 15, "subsample": 0.9},
    {"depth": 8, "learning_rate": 0.04, "l2_leaf_reg": 10.0, "min_data_in_leaf": 30, "subsample": 0.85},
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3.0, "min_data_in_leaf": 30, "subsample": 0.85},
    {"depth": 7, "learning_rate": 0.06, "l2_leaf_reg": 4.0, "min_data_in_leaf": 20, "subsample": 0.8},
]

tuning_results = []

for i, cfg_ in enumerate(search_space, 1):
    params = base_params.copy()
    params.update(cfg_)

    print(f"\n=== Config {i}/{len(search_space)}: {cfg_} ===")
    mean, std, oof_f1, scores = catboost_cv_score(X_fe, y, cat_feature_indices_fe, params, verbose_eval=0)

    tuning_results.append({
        "config_id": i,
        "params": cfg_,
        "mean_f1": mean,
        "std_f1": std,
        "oof_f1": oof_f1
    })

tuning_df = pd.DataFrame(tuning_results).sort_values("oof_f1", ascending=False)
tuning_df


=== Config 1/6: {'depth': 6, 'learning_rate': 0.06, 'l2_leaf_reg': 6.0, 'min_data_in_leaf': 20, 'subsample': 0.9} ===
Fold 1: 0.8037
Fold 2: 0.8362
Fold 3: 0.7981
Fold 4: 0.7923
Fold 5: 0.8065
Mean: 0.8074 ± 0.0152 | OOF: 0.8077

=== Config 2/6: {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 8.0, 'min_data_in_leaf': 25, 'subsample': 0.9} ===
Fold 1: 0.8078
Fold 2: 0.8384
Fold 3: 0.7982
Fold 4: 0.8112
Fold 5: 0.8056
Mean: 0.8122 ± 0.0137 | OOF: 0.8126

=== Config 3/6: {'depth': 5, 'learning_rate': 0.08, 'l2_leaf_reg': 5.0, 'min_data_in_leaf': 15, 'subsample': 0.9} ===
Fold 1: 0.8000
Fold 2: 0.8344
Fold 3: 0.7931
Fold 4: 0.8043
Fold 5: 0.8051
Mean: 0.8074 ± 0.0142 | OOF: 0.8078

=== Config 4/6: {'depth': 8, 'learning_rate': 0.04, 'l2_leaf_reg': 10.0, 'min_data_in_leaf': 30, 'subsample': 0.85} ===
Fold 1: 0.8152
Fold 2: 0.8300
Fold 3: 0.7899
Fold 4: 0.8092
Fold 5: 0.8037
Mean: 0.8096 ± 0.0132 | OOF: 0.8100

=== Config 5/6: {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3.0, 'min_

,config_id,params,mean_f1,std_f1,oof_f1
1,2,"{'depth': 7, 'learning_rate': 0.05, 'l2_leaf_r...",0.812240,0.013746,0.812602
4,5,"{'depth': 6, 'learning_rate': 0.05, 'l2_leaf_r...",0.810299,0.012162,0.810484
5,6,"{'depth': 7, 'learning_rate': 0.06, 'l2_leaf_r...",0.809833,0.012819,0.810127
3,4,"{'depth': 8, 'learning_rate': 0.04, 'l2_leaf_r...",0.809603,0.013217,0.810039
2,3,"{'depth': 5, 'learning_rate': 0.08, 'l2_leaf_r...",0.807383,0.014172,0.807787
0,1,"{'depth': 6, 'learning_rate': 0.06, 'l2_leaf_r...",0.807381,0.015226,0.807713
